# Spectra with the probe bands overlaid (D244, D270)

Look at the spectra of a dataset's windows, per class, with the spectral probe's bands drawn on top, to
judge visually whether the bands make sense. The windows are rebuilt exactly as the encoders received them
(`encoder_inputs()` of `probe_spectral.py`: the run's windows, resampling and standardisation), from the
first run of the dataset's config in `configs/` (Paderborn: the first run of the chosen combination).
The bands are the probe's: `n_bands` equal-width bands from 0 to Nyquist, DC bin left out; each band is
labelled with its number and, where `results/<dataset>_eta2.csv` exists, its η² (D270).

Set the parameters in the next cell and run all. Run from the repo root folder. Clear the outputs before committing.

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "spectral_probe":  # opened from its own folder: paths are relative to the repo root
    os.chdir(os.path.join("..", ".."))
sys.path.insert(0, "studies/spectral_probe")
from harness import runs
from probe_spectral import encoder_inputs, run_config

DATASET = "cwru"     # a config name in studies/spectral_probe/configs/: forda, cwru, hust, paderborn (xjtu_sy: no runs yet)
COMBINATION = 0      # Paderborn only: combination 0..9 (each has its own training bearings)
SPLIT = "train"      # "train" or "test" windows
LOG_FREQ = False     # logarithmic frequency axis
N_EXAMPLES = 3       # single windows per class in the last figure
CHUNK = 256          # windows per FFT call (memory only; does not change any value)

## The windows

In [ ]:
def pick_run(dataset, combination):
    """First run in the dataset's spectral-probe config that has classes (and is of the combination, for Paderborn)."""
    study = runs.load_config(f"studies/spectral_probe/configs/{dataset}.yaml")
    for folder in study["runs"]:
        folder = folder.rstrip("/")
        if run_config(folder).get("task") == "regression":  # e.g. XJTU-SY RUL: no classes
            continue
        split_path = os.path.join(folder, "split.json")
        k = json.load(open(split_path))["combination"] if os.path.exists(split_path) else None
        if k is None or k == combination:
            return folder, study["n_bands"], k
    raise ValueError(f"{dataset}: no run with classes (combination {combination}) in its config")


folder, n_bands, combination = pick_run(DATASET, COMBINATION)
X, _, cfg = encoder_inputs(folder)
X = X[SPLIT]  # (windows, length, channels)
channels = cfg["channels"]
rate = cfg.get("windows", {}).get("sample_rate_hz")  # FordA: none given -> cycles per sample

# class of each window: the label column of windows.csv, else the run's class numbers (FordA)
table = pd.read_csv(os.path.join(folder, "windows.csv"), dtype={"run_id": str, "capture_id": str, "group": str})
rows = table[table.split == SPLIT].sort_values("row")
if "label" in rows:
    labels = rows.label.astype(str).to_numpy()
else:
    labels = np.array([f"class {y}" for y in np.load(os.path.join(folder, "seed_0", f"y_{SPLIT}.npy"))])
classes = sorted(np.unique(labels))

# power per window, bin and channel, DC bin left out (as the targets, D244); float32 to save memory
f = np.fft.rfftfreq(X.shape[1])[1:]  # cycles per sample
P = np.empty((len(X), len(f), len(channels)), dtype=np.float32)
for i in range(0, len(X), CHUNK):
    P[i:i + CHUNK] = np.abs(np.fft.rfft(X[i:i + CHUNK], axis=1))[:, 1:] ** 2

# the probe's bands: band b holds edges[b] < f <= edges[b + 1] (probe_spectral.spectral_targets)
edges = np.linspace(0.0, 0.5, n_bands + 1)
band = np.searchsorted(edges, f, side="left") - 1
E = np.stack([P[:, band == b].sum(axis=1) for b in range(n_bands)], axis=1)  # band energy (windows, bands, channels)
relative = np.log(E / E.sum(axis=1, keepdims=True))  # the relative target: log share of the total energy

scale = rate if rate else 1.0  # x axis in Hz where the sample rate is known
unit = "Hz" if rate else "cycles per sample"
x, edges_x = f * scale, edges * scale

# η² per band and channel (D270), if computed for this dataset
eta_path = f"studies/spectral_probe/results/{DATASET}_eta2.csv"
eta = None
if os.path.exists(eta_path):
    e = pd.read_csv(eta_path)
    e = e[e.combination.fillna(-1) == (-1 if combination is None else combination)]
    eta = {ch: g.sort_values("band").eta2.to_numpy() for ch, g in e.groupby("channel")}

print(f"run: {folder} ({SPLIT} windows{'' if combination is None else f', combination {combination}'})")
print(f"{len(X)} windows of {X.shape[1]} samples, channels {channels}, sample rate {rate or 'unknown'}")
print(f"bin width {f[0] * scale:.4g} {unit}; band width {edges_x[1]:.4g} {unit} ({(band == 1).sum()} bins)")
print(pd.Series(labels).value_counts().sort_index().to_string())

## Class spectra

Median log power per class over the windows (shading: interquartile range), with the bands: alternate grey
shading, band number at the top and η² underneath (D270).

In [ ]:
colours = {c: plt.cm.tab10(i % 10) for i, c in enumerate(classes)}

def draw_bands(ax, ch):
    """Shade every other band and write its number (and η²) at the top of the axes."""
    for b in range(n_bands):
        if b % 2 == 0:
            ax.axvspan(edges_x[b], edges_x[b + 1], color="grey", alpha=0.12, lw=0)
        label = f"{b}" if eta is None else f"{b}\n{eta[ch][b]:.2f}"
        ax.text((edges_x[b] + edges_x[b + 1]) / 2, 1.01, label, transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=7)
    if LOG_FREQ:
        ax.set_xscale("log")

for c, ch in enumerate(channels):
    fig, ax = plt.subplots(figsize=(14, 5))
    for k in classes:
        logp = np.log10(P[labels == k, :, c])
        q25, q50, q75 = np.percentile(logp, [25, 50, 75], axis=0)
        ax.plot(x, q50, color=colours[k], lw=1, label=f"{k} ({(labels == k).sum()})")
        ax.fill_between(x, q25, q75, color=colours[k], alpha=0.15, lw=0)
    draw_bands(ax, ch)
    ax.set(xlim=(x[0] if LOG_FREQ else 0, x[-1]), xlabel=f"frequency ({unit})", ylabel="log10 power")
    ax.set_title(f"{DATASET}, {ch}: median and IQR per class" + ("" if eta is None else " (top: band, η²)"), pad=28)
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## The targets per class

What the probes predict: the log relative energy of each band (log share of the total), per class, mean
± one standard deviation over the windows.

In [ ]:
centre = (edges_x[:-1] + edges_x[1:]) / 2
for c, ch in enumerate(channels):
    fig, ax = plt.subplots(figsize=(14, 4))
    for k in classes:
        r = relative[labels == k, :, c]
        m, s = r.mean(axis=0), r.std(axis=0)
        ax.plot(centre, m, marker="o", ms=3, color=colours[k], label=k)
        ax.fill_between(centre, m - s, m + s, color=colours[k], alpha=0.15, lw=0)
    draw_bands(ax, ch)
    ax.set(xlim=(edges_x[1] if LOG_FREQ else 0, edges_x[-1]), xlabel=f"band centre ({unit})",
           ylabel="log relative band energy")
    ax.set_title(f"{DATASET}, {ch}: target per band and class", pad=28)
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## Single windows

`N_EXAMPLES` random windows of every class (thin lines) on top of the class median (thick), one panel per
class, first channel only (change `c` for another).

In [ ]:
c = 0  # channel index
rng = np.random.default_rng(0)
ncol = 2
nrow = int(np.ceil(len(classes) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(14, 3 * nrow), sharex=True, sharey=True, squeeze=False)
for ax, k in zip(axes.flat, classes):
    idx = np.flatnonzero(labels == k)
    for i in rng.choice(idx, size=min(N_EXAMPLES, len(idx)), replace=False):
        ax.plot(x, np.log10(P[i, :, c]), lw=0.5, alpha=0.6, color=colours[k])
    ax.plot(x, np.median(np.log10(P[idx, :, c]), axis=0), lw=1.5, color="black")
    for b in range(1, n_bands):
        ax.axvline(edges_x[b], color="grey", lw=0.5)
    if LOG_FREQ:
        ax.set_xscale("log")
    ax.set_title(f"{k} ({len(idx)} windows)", fontsize=9)
for ax in axes.flat[len(classes):]:
    ax.set_visible(False)
fig.supxlabel(f"frequency ({unit})")
fig.supylabel("log10 power")
fig.suptitle(f"{DATASET}, {channels[c]}: single windows and class median")
plt.tight_layout()
plt.show()